# 🧬 Real-World NGS Pipeline: NCBI SRA → Final Results

<div style='background:#E3F2FD;padding:15px;border-radius:8px;border-left:5px solid #1976D2'>
<b>📌 Dataset:</b> SRR1518253 — Human WES (1000 Genomes Project, HG00096, British male)<br>
<b>📌 Reference:</b> hg38 chromosome 22 (fast for Colab free tier)<br>
<b>📌 Runtime:</b> ~30–45 minutes on Colab free tier<br>
<b>📌 Output:</b> QC reports, aligned BAM, filtered VCF, annotated CSV, dashboards
</div>

---

## 🗺️ Pipeline Overview

```
NCBI SRA Download → QC (FastQC) → Trim (Trimmomatic) → Align (BWA-MEM)
     → Dedup (Picard) → Variant Call (GATK) → Filter → Annotate → Dashboard
```

| Step | Stage | Tool | Expected Output |
|------|-------|------|-----------------|
| 0 | Install tools | apt-get / pip | All tools ready |
| 1 | Setup directories | Python | Project structure |
| 2 | Download NCBI data | fasterq-dump | FASTQ.gz files |
| 3 | Download reference | wget + BWA index | hg38 chr22 + index |
| 4 | Quality control | FastQC | QC metrics + plots |
| 5 | Read trimming | Trimmomatic | Cleaned FASTQ |
| 6 | Sequence alignment | BWA-MEM | Sorted BAM |
| 7 | Mark duplicates | Picard | Dedup BAM |
| 8 | Variant calling | GATK HaplotypeCaller | Raw VCF |
| 9 | Variant filtering | GATK VariantFiltration | Filtered VCF |
| 10 | Parse & annotate | BCFtools + Python | Annotated CSV |
| 11 | Visualization | Matplotlib | Final dashboards |
| 12 | Summary report | Python | Full report |
| 13 | Download results | Colab files | ZIP archive |


---
## ⚙️ STEP 0 — Install All Bioinformatics Tools

In [ ]:
%%bash
set -e
echo '============================================'
echo '   Installing All Bioinformatics Tools'
echo '============================================'

# System packages
apt-get update -qq
apt-get install -y -qq \
    fastqc trimmomatic bwa samtools bcftools \
    wget curl unzip default-jdk pigz 2>/dev/null
echo '✅ System packages installed'

# SRA Toolkit
cd /tmp
wget -q https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
tar -xzf sratoolkit.current-ubuntu64.tar.gz 2>/dev/null
SRADIR=$(ls -d /tmp/sratoolkit.*-ubuntu64 2>/dev/null | head -1)
[ -n "$SRADIR" ] && cp $SRADIR/bin/{prefetch,fastq-dump,fasterq-dump} /usr/local/bin/ 2>/dev/null || true
echo '✅ SRA Toolkit installed'

# Picard
wget -q https://github.com/broadinstitute/picard/releases/download/2.27.5/picard.jar \
    -O /usr/local/bin/picard.jar
printf '#!/bin/bash\njava -Xmx4g -jar /usr/local/bin/picard.jar "$@"\n' \
    > /usr/local/bin/picard && chmod +x /usr/local/bin/picard
echo '✅ Picard installed'

# GATK
cd /tmp
wget -q https://github.com/broadinstitute/gatk/releases/download/4.3.0.0/gatk-4.3.0.0.zip
unzip -q gatk-4.3.0.0.zip
cp /tmp/gatk-4.3.0.0/gatk /usr/local/bin/
cp /tmp/gatk-4.3.0.0/gatk-package-4.3.0.0-local.jar /usr/local/bin/
echo '✅ GATK 4.3 installed'

echo ''
echo '============================================'
echo '   ✅ ALL TOOLS INSTALLED SUCCESSFULLY'
echo '============================================'

In [ ]:
# Install Python packages
!pip install pandas numpy matplotlib seaborn biopython pysam -q

import subprocess, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import gzip, random
from datetime import datetime

print('🔧 Verifying tools:')
for tool in ['fastqc','trimmomatic','bwa','samtools','bcftools','picard','gatk']:
    exists = subprocess.run(['which', tool], capture_output=True).returncode == 0
    print(f"  {'✅' if exists else '❌'} {tool}")

print('\n✅ Environment ready!')

---
## 📁 STEP 1 — Project Configuration & Directory Setup

In [ ]:
# ================================================================
#  PROJECT CONFIGURATION — Modify these to use a different sample
# ================================================================
PROJECT_DIR  = '/content/ngs_project'
SRA_ID       = 'SRR1518253'          # NCBI SRA accession number
SAMPLE_NAME  = 'HG00096_chr22'       # Human-readable label
CHROM        = 'chr22'               # Chromosome to analyze
MAX_READS    = 500000                # Reads to download (None = all)
THREADS      = 2                     # Colab free tier = 2 CPUs
MIN_QUAL     = 30                    # Minimum variant QUAL score
MIN_DEPTH    = 10                    # Minimum read depth
# ================================================================

dirs = [
    'data/raw', 'reference',
    'results/fastqc', 'results/trimmed',
    'results/aligned', 'results/dedup',
    'results/variants', 'results/annotated',
    'results/plots', 'logs'
]
for d in dirs:
    os.makedirs(f'{PROJECT_DIR}/{d}', exist_ok=True)

print(f'📂 Project root : {PROJECT_DIR}')
print(f'🔬 SRA ID       : {SRA_ID}')
print(f'🧬 Sample name  : {SAMPLE_NAME}')
print(f'📌 Chromosome   : {CHROM}')
print(f'🧵 Threads      : {THREADS}')
print('\n✅ Directory structure created!')

---
## 📥 STEP 2 — Download Real FASTQ Data from NCBI SRA

> **SRR1518253** — Whole Exome Sequencing of HG00096 (1000 Genomes Project).  
> We limit to 500 k read-pairs so the notebook finishes within Colab's free-tier time limit.  
> Remove `-X` to download the complete dataset.

In [ ]:
%%bash -s "$PROJECT_DIR" "$SRA_ID" "$SAMPLE_NAME" "$MAX_READS" "$THREADS"
PROJECT_DIR=$1; SRA_ID=$2; SAMPLE_NAME=$3; MAX_READS=$4; THREADS=$5

echo '============================================'
echo "  Downloading $SRA_ID from NCBI SRA"
echo '============================================'

# Configure SRA toolkit (disable interactive prompts)
mkdir -p ~/.ncbi
cat > ~/.ncbi/user-settings.mkfg << 'EOF'
/LIBS/GUID = "colab-ngs-pipeline-demo-2024"
/repository/user/main/public/root = "/tmp/sra-cache"
/tools/prefetch/max_size = "30g"
EOF

mkdir -p /tmp/sra-cache
cd $PROJECT_DIR/data/raw

echo "📥 Downloading up to $MAX_READS reads..."
fasterq-dump $SRA_ID \
    --split-files \
    --outdir $PROJECT_DIR/data/raw \
    --threads $THREADS \
    -X $MAX_READS \
    --progress \
    --temp /tmp \
    2>&1 | grep -E 'spots|reads|written|err|Err' || true

# Compress
echo '🗜️  Compressing...'
[ -f $PROJECT_DIR/data/raw/${SRA_ID}_1.fastq ] && \
    pigz -p $THREADS $PROJECT_DIR/data/raw/${SRA_ID}_1.fastq
[ -f $PROJECT_DIR/data/raw/${SRA_ID}_2.fastq ] && \
    pigz -p $THREADS $PROJECT_DIR/data/raw/${SRA_ID}_2.fastq

# Rename
mv $PROJECT_DIR/data/raw/${SRA_ID}_1.fastq.gz \
   $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R1.fastq.gz 2>/dev/null || true
mv $PROJECT_DIR/data/raw/${SRA_ID}_2.fastq.gz \
   $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R2.fastq.gz 2>/dev/null || true

echo ''
echo '📊 Downloaded files:'
ls -lh $PROJECT_DIR/data/raw/*.fastq.gz 2>/dev/null

echo ''
echo '📊 Read counts:'
for f in $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R*.fastq.gz; do
    N=$(zcat $f | wc -l)
    echo "  $(basename $f): $((N/4)) reads"
done

echo ''
echo '✅ Download complete!'

---
## 🧬 STEP 3 — Download & Index hg38 chr22 Reference Genome

In [ ]:
%%bash -s "$PROJECT_DIR" "$THREADS"
PROJECT_DIR=$1; THREADS=$2
REF=$PROJECT_DIR/reference/chr22.fa

echo '============================================'
echo '  Downloading hg38 chr22 Reference Genome'
echo '============================================'

# Download chr22 (~52 MB, much faster than full genome ~3 GB)
echo '📥 Downloading chr22 from UCSC...'
wget -q --show-progress \
    https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz \
    -O $PROJECT_DIR/reference/chr22.fa.gz

echo '🗜️  Decompressing...'
gunzip $PROJECT_DIR/reference/chr22.fa.gz
echo "   Size: $(du -sh $REF | cut -f1)"

# BWA index (~3 min)
echo '📇 BWA index...'
bwa index $REF 2>&1 | grep -E 'index|Finished|error' | tail -3

# SAMtools FASTA index
echo '📇 SAMtools fai index...'
samtools faidx $REF

# GATK sequence dictionary
echo '📇 GATK sequence dictionary...'
picard CreateSequenceDictionary R=$REF O=$PROJECT_DIR/reference/chr22.dict \
    2>&1 | grep -E 'Done|ERROR' | tail -2

echo ''
echo '📂 Reference files:'
ls -lh $PROJECT_DIR/reference/

echo ''
echo '✅ Reference genome ready!'

---
## 🔍 STEP 4 — Quality Control (FastQC + Python Dashboard)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME" "$THREADS"
PROJECT_DIR=$1; SAMPLE_NAME=$2; THREADS=$3

echo '============================================'
echo '  Running FastQC'
echo '============================================'

fastqc \
    $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R1.fastq.gz \
    $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R2.fastq.gz \
    --outdir $PROJECT_DIR/results/fastqc \
    --threads $THREADS \
    2>&1 | grep -E 'Analysis|Complete|Started|Failed'

echo ''
ls -lh $PROJECT_DIR/results/fastqc/
echo '✅ FastQC complete!'

In [ ]:
def parse_fastq_metrics(filepath, label, max_reads=10000):
    """Parse real FASTQ for QC metrics."""
    qual_scores, gc_contents, read_lengths = [], [], {}
    per_base = {}
    count = 0
    with gzip.open(filepath, 'rt') as fh:
        while count < max_reads:
            rid  = fh.readline()
            if not rid: break
            seq  = fh.readline().strip()
            _    = fh.readline()
            qual = fh.readline().strip()
            if not qual: break
            scores = [ord(c)-33 for c in qual]
            qual_scores.append(np.mean(scores))
            gc = (seq.count('G')+seq.count('C')) / max(len(seq),1) * 100
            gc_contents.append(gc)
            rl = len(seq)
            read_lengths[rl] = read_lengths.get(rl, 0) + 1
            for pos, s in enumerate(scores):
                per_base.setdefault(pos, []).append(s)
            count += 1
    return dict(
        label=label, n=count,
        mean_q=np.mean(qual_scores),
        q30=sum(q>=30 for q in qual_scores)/max(count,1)*100,
        mean_len=np.mean(list(read_lengths.keys())),
        mean_gc=np.mean(gc_contents),
        qual_scores=qual_scores,
        gc_contents=gc_contents,
        per_base=per_base
    )

print('📊 Parsing real FASTQ metrics...\n')
r1 = parse_fastq_metrics(f'{PROJECT_DIR}/data/raw/{SAMPLE_NAME}_R1.fastq.gz', 'R1')
r2 = parse_fastq_metrics(f'{PROJECT_DIR}/data/raw/{SAMPLE_NAME}_R2.fastq.gz', 'R2')

for m in [r1, r2]:
    status = '✅ PASS' if m['mean_q'] >= 25 and m['q30'] >= 60 else '⚠️ REVIEW'
    print(f"  {m['label']}: reads={m['n']:,}  meanQ={m['mean_q']:.1f}  "
          f"Q30={m['q30']:.1f}%  GC={m['mean_gc']:.1f}%  "
          f"len={m['mean_len']:.0f}bp  [{status}]")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle(f'QC Dashboard  |  {SAMPLE_NAME}  |  Source: NCBI {SRA_ID}',
             fontsize=13, fontweight='bold')
C = {'R1':'#1976D2', 'R2':'#E53935'}

for ax, m, col in zip(axes[0,:2], [r1,r2], [C['R1'],C['R2']]):
    pos = sorted(m['per_base'])[:120]
    mn  = [np.mean(m['per_base'][p]) for p in pos]
    sd  = [np.std (m['per_base'][p]) for p in pos]
    ax.fill_between(pos,
                    [a-b for a,b in zip(mn,sd)],
                    [a+b for a,b in zip(mn,sd)],
                    alpha=0.15, color=col)
    ax.plot(pos, mn, color=col, lw=1.8)
    ax.axhline(30, color='red',    ls='--', lw=1, label='Q30')
    ax.axhline(20, color='orange', ls='--', lw=1, label='Q20')
    ax.set(title=f'Per-Base Quality ({m["label"]})', ylim=(0,42),
           xlabel='Base Position', ylabel='Phred Score')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

ax = axes[0,2]
for m,col in zip([r1,r2],[C['R1'],C['R2']]):
    ax.hist(m['gc_contents'], bins=40, alpha=0.55, color=col,
            label=m['label'], edgecolor='white')
ax.axvline(50, color='black', ls='--', lw=1.5, label='Expected 50%')
ax.set(title='GC Content Distribution', xlabel='GC (%)', ylabel='Reads')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

ax = axes[1,0]
for m,col in zip([r1,r2],[C['R1'],C['R2']]):
    ax.hist(m['qual_scores'], bins=30, alpha=0.65, color=col,
            label=m['label'], edgecolor='white')
ax.axvline(30, color='red', ls='--', lw=1.5, label='Q30')
ax.set(title='Mean Quality Distribution', xlabel='Mean Phred', ylabel='Reads')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

ax = axes[1,1]
metrics  = ['Mean Quality','Q30 (%)','GC Content (%)']
r1_vals  = [r1['mean_q'], r1['q30'], r1['mean_gc']]
r2_vals  = [r2['mean_q'], r2['q30'], r2['mean_gc']]
x = np.arange(len(metrics)); w = 0.35
ax.bar(x-w/2, r1_vals, w, label='R1', color=C['R1'], edgecolor='black', lw=0.5)
ax.bar(x+w/2, r2_vals, w, label='R2', color=C['R2'], edgecolor='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=9)
ax.set_title('QC Metric Comparison'); ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.25)

ax = axes[1,2]; ax.axis('off')
rows  = [['Metric','R1','R2','Threshold','Pass?'],
         ['Mean Q', f"{r1['mean_q']:.1f}", f"{r2['mean_q']:.1f}", '≥ 25',
          '✅' if r1['mean_q']>=25 else '❌'],
         ['Q30 (%)', f"{r1['q30']:.1f}", f"{r2['q30']:.1f}", '≥ 60%',
          '✅' if r1['q30']>=60 else '❌'],
         ['GC (%)', f"{r1['mean_gc']:.1f}", f"{r2['mean_gc']:.1f}", '40–60',
          '✅' if 35<=r1['mean_gc']<=65 else '⚠️'],
         ['Reads', f"{r1['n']:,}", f"{r2['n']:,}", '—','—']]
tbl = ax.table(cellText=rows[1:], colLabels=rows[0],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.9)
for j in range(5):
    tbl[0,j].set_facecolor('#1565C0')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,5):
    for j in range(5):
        tbl[i,j].set_facecolor('#E3F2FD' if i%2==0 else 'white')
ax.set_title('QC Rubric Summary', fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/plots/01_QC_Dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ QC Dashboard saved → results/plots/01_QC_Dashboard.png')

---
## ✂️ STEP 5 — Read Trimming (Trimmomatic)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME" "$THREADS"
PROJECT_DIR=$1; SAMPLE_NAME=$2; THREADS=$3

echo '============================================'
echo '  Trimmomatic — Adapter & Quality Trimming'
echo '============================================'

# Find or create adapter file
ADAPTERS=$(find / -name 'TruSeq3-PE-2.fa' 2>/dev/null | head -1)
if [ -z "$ADAPTERS" ]; then
    cat > /tmp/adapters.fa << 'ADAPT'
>PrefixPE/1
TACACTCTTTCCCTACACGACGCTCTTCCGATCT
>PrefixPE/2
GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT
>PE1
TACACTCTTTCCCTACACGACGCTCTTCCGATCT
>PE2
GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT
ADAPT
    ADAPTERS=/tmp/adapters.fa
fi

trimmomatic PE \
    $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R1.fastq.gz \
    $PROJECT_DIR/data/raw/${SAMPLE_NAME}_R2.fastq.gz \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R1_paired.fastq.gz \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R1_unpaired.fastq.gz \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R2_paired.fastq.gz \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R2_unpaired.fastq.gz \
    ILLUMINACLIP:${ADAPTERS}:2:30:10:2:keepBothReads \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36 \
    -threads $THREADS \
    2>&1 | grep -E 'Input|Both|Dropped|Surviving'

echo ''
echo '📂 Trimmed output:'
ls -lh $PROJECT_DIR/results/trimmed/*_paired.fastq.gz
echo ''
echo '✅ Trimming complete!'

---
## 🗺️ STEP 6 — Sequence Alignment (BWA-MEM → SAMtools sort)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME" "$THREADS"
PROJECT_DIR=$1; SAMPLE_NAME=$2; THREADS=$3
REF=$PROJECT_DIR/reference/chr22.fa
BAM=$PROJECT_DIR/results/aligned/${SAMPLE_NAME}.bam

echo '============================================'
echo '  BWA-MEM Alignment → hg38 chr22'
echo '============================================'

# Align with read-group tags (required for GATK)
bwa mem \
    -t $THREADS \
    -M \
    -R "@RG\tID:${SAMPLE_NAME}\tSM:HG00096\tPL:ILLUMINA\tLB:WES_lib1\tPU:${SAMPLE_NAME}" \
    $REF \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R1_paired.fastq.gz \
    $PROJECT_DIR/results/trimmed/${SAMPLE_NAME}_R2_paired.fastq.gz \
    2>$PROJECT_DIR/logs/bwa.log \
    | samtools sort -@ $THREADS -o $BAM

# Index BAM
samtools index $BAM

echo ''
echo '📊 Flagstat (alignment statistics):'
samtools flagstat $BAM | tee $PROJECT_DIR/results/aligned/${SAMPLE_NAME}_flagstat.txt

echo ''
echo '📊 Coverage summary (chr22):'
samtools coverage $BAM | head -3

echo ''
echo '✅ Alignment complete!'

---
## 🔁 STEP 7 — Mark PCR Duplicates (Picard)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME"
PROJECT_DIR=$1; SAMPLE_NAME=$2

echo '============================================'
echo '  Picard MarkDuplicates'
echo '============================================'

picard MarkDuplicates \
    INPUT=$PROJECT_DIR/results/aligned/${SAMPLE_NAME}.bam \
    OUTPUT=$PROJECT_DIR/results/dedup/${SAMPLE_NAME}_dedup.bam \
    METRICS_FILE=$PROJECT_DIR/results/dedup/${SAMPLE_NAME}_dup_metrics.txt \
    REMOVE_DUPLICATES=false \
    CREATE_INDEX=true \
    VALIDATION_STRINGENCY=SILENT \
    2>&1 | grep -E 'INFO MarkDuplicates|Done|pairs|singletons' | tail -6

echo ''
echo '📊 Duplication metrics:'
grep -A3 'ESTIMATED_LIBRARY_SIZE' \
    $PROJECT_DIR/results/dedup/${SAMPLE_NAME}_dup_metrics.txt 2>/dev/null | head -4

echo ''
echo '✅ Duplicate marking complete!'

---
## 🧬 STEP 8 — Variant Calling (GATK HaplotypeCaller)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME"
PROJECT_DIR=$1; SAMPLE_NAME=$2
REF=$PROJECT_DIR/reference/chr22.fa

echo '============================================'
echo '  GATK HaplotypeCaller — Variant Calling'
echo '============================================'

gatk HaplotypeCaller \
    -I  $PROJECT_DIR/results/dedup/${SAMPLE_NAME}_dedup.bam \
    -R  $REF \
    -O  $PROJECT_DIR/results/variants/${SAMPLE_NAME}_raw.vcf.gz \
    --sample-name HG00096 \
    --native-pair-hmm-threads 2 \
    2>&1 | grep -E 'INFO HaplotypeCaller.*done|ProgressMeter' | tail -4

echo ''
echo '📊 Raw variant counts:'
TOTAL=$(bcftools view $PROJECT_DIR/results/variants/${SAMPLE_NAME}_raw.vcf.gz | grep -vc '^#')
SNPS=$(bcftools view --type snps $PROJECT_DIR/results/variants/${SAMPLE_NAME}_raw.vcf.gz | grep -vc '^#')
INDELS=$(bcftools view --type indels $PROJECT_DIR/results/variants/${SAMPLE_NAME}_raw.vcf.gz | grep -vc '^#')
echo "  Total : $TOTAL"
echo "  SNPs  : $SNPS"
echo "  INDELs: $INDELS"

echo ''
echo '✅ Variant calling complete!'

---
## 🔬 STEP 9 — Variant Filtering (GATK Best Practices)

In [ ]:
%%bash -s "$PROJECT_DIR" "$SAMPLE_NAME"
PROJECT_DIR=$1; SAMPLE_NAME=$2
REF=$PROJECT_DIR/reference/chr22.fa
RAW=$PROJECT_DIR/results/variants/${SAMPLE_NAME}_raw.vcf.gz
VDIR=$PROJECT_DIR/results/variants

echo '============================================'
echo '  GATK Hard Filtering (SNPs + INDELs)'
echo '============================================'

# ------- SNP hard filters -------
echo '🔹 Selecting SNPs...'
gatk SelectVariants -R $REF -V $RAW \
    --select-type-to-include SNP \
    -O $VDIR/${SAMPLE_NAME}_snps.vcf.gz 2>/dev/null

echo '🔹 Applying SNP filters...'
gatk VariantFiltration -R $REF \
    -V $VDIR/${SAMPLE_NAME}_snps.vcf.gz \
    --filter-expression 'QD < 2.0'          --filter-name 'QD2'   \
    --filter-expression 'FS > 60.0'          --filter-name 'FS60'  \
    --filter-expression 'MQ < 40.0'          --filter-name 'MQ40'  \
    --filter-expression 'MQRankSum < -12.5'  --filter-name 'MQRankSum125' \
    --filter-expression 'ReadPosRankSum < -8.0' --filter-name 'ReadPosRS8' \
    -O $VDIR/${SAMPLE_NAME}_snps_filtered.vcf.gz 2>/dev/null

# ------- INDEL hard filters -------
echo '🔸 Selecting INDELs...'
gatk SelectVariants -R $REF -V $RAW \
    --select-type-to-include INDEL \
    -O $VDIR/${SAMPLE_NAME}_indels.vcf.gz 2>/dev/null

echo '🔸 Applying INDEL filters...'
gatk VariantFiltration -R $REF \
    -V $VDIR/${SAMPLE_NAME}_indels.vcf.gz \
    --filter-expression 'QD < 2.0'     --filter-name 'QD2'   \
    --filter-expression 'FS > 200.0'   --filter-name 'FS200' \
    --filter-expression 'ReadPosRankSum < -20.0' --filter-name 'ReadPosRS20' \
    -O $VDIR/${SAMPLE_NAME}_indels_filtered.vcf.gz 2>/dev/null

echo ''
echo '📊 Variants passing filters:'
SNP_PASS=$(bcftools view -f PASS $VDIR/${SAMPLE_NAME}_snps_filtered.vcf.gz | grep -vc '^#')
IND_PASS=$(bcftools view -f PASS $VDIR/${SAMPLE_NAME}_indels_filtered.vcf.gz | grep -vc '^#')
echo "  SNPs   PASS : $SNP_PASS"
echo "  INDELs PASS : $IND_PASS"
echo "  Total  PASS : $((SNP_PASS + IND_PASS))"

echo ''
echo '✅ Filtering complete!'

---
## 📝 STEP 10 — Parse VCF → Annotated DataFrame

In [ ]:
def vcf_to_df(vcf_path, vtype):
    """Parse a PASS-filtered VCF into a tidy DataFrame."""
    res = subprocess.run(
        ['bcftools', 'view', '-f', 'PASS', vcf_path],
        capture_output=True, text=True
    )
    rows = []
    for line in res.stdout.splitlines():
        if line.startswith('#') or not line: continue
        p = line.split('\t')
        if len(p) < 8: continue
        chrom,pos,vid,ref,alt,qual,flt,info = p[:8]
        idict = dict(kv.split('=',1) for kv in info.split(';') if '=' in kv)
        try: q = float(qual)
        except: q = None
        rows.append(dict(
            CHROM=chrom, POS=int(pos), ID=vid,
            REF=ref, ALT=alt, QUAL=q, FILTER=flt,
            TYPE=vtype,
            DP=int(idict.get('DP',0)),
            QD=float(idict.get('QD',0)),
            FS=float(idict.get('FS',0)),
            MQ=float(idict.get('MQ',0)),
        ))
    return pd.DataFrame(rows)

VDIR = f'{PROJECT_DIR}/results/variants'
print('📊 Parsing filtered VCFs...\n')

snp_df   = vcf_to_df(f'{VDIR}/{SAMPLE_NAME}_snps_filtered.vcf.gz',   'SNP')
indel_df = vcf_to_df(f'{VDIR}/{SAMPLE_NAME}_indels_filtered.vcf.gz', 'INDEL')
all_df   = pd.concat([snp_df, indel_df], ignore_index=True)

# Additional Python-level filters
filt_df = all_df[(all_df['QUAL'] >= MIN_QUAL) & (all_df['DP'] >= MIN_DEPTH)].copy()

# Classify by QUAL
filt_df['Confidence'] = pd.cut(
    filt_df['QUAL'],
    bins=[0, 50, 100, float('inf')],
    labels=['Low (30-50)', 'Medium (50-100)', 'High (>100)']
)

print(f'  Raw variants total : {len(all_df):,}')
print(f'  After Py filters   : {len(filt_df):,}')
print(f'  SNPs               : {len(filt_df[filt_df["TYPE"]=="SNP"]):,}')
print(f'  INDELs             : {len(filt_df[filt_df["TYPE"]=="INDEL"]):,}')
print(f'  Median depth       : {filt_df["DP"].median():.0f}x')
print(f'  Median QUAL        : {filt_df["QUAL"].median():.1f}')

filt_df.to_csv(f'{PROJECT_DIR}/results/annotated/{SAMPLE_NAME}_variants.csv', index=False)
print(f'\n✅ Saved → results/annotated/{SAMPLE_NAME}_variants.csv')
print('\n📋 Preview:')
print(filt_df[['CHROM','POS','REF','ALT','TYPE','QUAL','DP','Confidence']].head(10).to_string(index=False))

---
## 📊 STEP 11 — Final Results Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    f'Variant Analysis Dashboard  |  HG00096  |  NCBI {SRA_ID}  |  hg38 {CHROM}',
    fontsize=14, fontweight='bold'
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# 1 — SNP vs INDEL pie
ax1 = fig.add_subplot(gs[0,0])
tc = filt_df['TYPE'].value_counts()
wedges,_,at = ax1.pie(
    tc.values, labels=tc.index,
    autopct='%1.1f%%',
    colors=['#42A5F5','#EF5350'],
    startangle=90,
    wedgeprops=dict(edgecolor='white', lw=2)
)
for a in at: a.set_fontweight('bold')
ax1.set_title(f'Variant Types\n(n={len(filt_df):,})', fontweight='bold')

# 2 — QUAL distribution
ax2 = fig.add_subplot(gs[0,1])
for vt, col in [('SNP','#42A5F5'),('INDEL','#EF5350')]:
    sub = filt_df[filt_df['TYPE']==vt]['QUAL'].dropna()
    if len(sub): ax2.hist(sub, bins=35, alpha=0.65, color=col, label=vt, edgecolor='white')
ax2.axvline(MIN_QUAL, color='red', ls='--', lw=2, label=f'Min QUAL={MIN_QUAL}')
ax2.set(title='Variant Quality (QUAL)', xlabel='QUAL Score', ylabel='Count')
ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

# 3 — Read depth distribution
ax3 = fig.add_subplot(gs[0,2])
dp = filt_df['DP'].clip(upper=300)
ax3.hist(dp, bins=40, color='#388E3C', edgecolor='white', alpha=0.8)
med = filt_df['DP'].median()
ax3.axvline(MIN_DEPTH, color='red',    ls='--', lw=1.5, label=f'Min depth={MIN_DEPTH}')
ax3.axvline(med,       color='orange', ls='-',  lw=2,   label=f'Median={med:.0f}x')
ax3.set(title='Read Depth (DP)', xlabel='Depth', ylabel='Count')
ax3.legend(fontsize=8); ax3.grid(alpha=0.25)

# 4 — Chromosomal scatter
ax4 = fig.add_subplot(gs[1,0:2])
for vt,col,y in [('SNP','#42A5F5',1.0),('INDEL','#EF5350',1.3)]:
    pos = filt_df[filt_df['TYPE']==vt]['POS']
    ax4.scatter(pos/1e6, [y]*len(pos), alpha=0.25, s=4, color=col, label=f'{vt} (n={len(pos):,})')
ax4.set(xlabel='Position on chr22 (Mb)',
        title=f'Variant Distribution — {CHROM}')
ax4.set_yticks([1.0,1.3]); ax4.set_yticklabels(['SNPs','INDELs'])
ax4.legend(loc='upper right', fontsize=9)
ax4.grid(alpha=0.2)

# 5 — Confidence tier bar
ax5 = fig.add_subplot(gs[1,2]); ax5.axis('off')

# Parse flagstat
flagfile = f'{PROJECT_DIR}/results/aligned/{SAMPLE_NAME}_flagstat.txt'
try:
    with open(flagfile) as fh: fl = fh.readlines()
    total_r = fl[0].split()[0]
    mapped  = fl[4].split()[0]
    map_pct = fl[4].split('(')[1].split('%')[0] if '(' in fl[4] else 'N/A'
except: total_r = mapped = map_pct = 'N/A'

dupfile = f'{PROJECT_DIR}/results/dedup/{SAMPLE_NAME}_dup_metrics.txt'
dup_pct = 'N/A'
try:
    with open(dupfile) as fh:
        for line in fh:
            parts = line.strip().split('\t')
            if len(parts) > 8 and parts[0] not in ('##','LIBRARY'):
                try: dup_pct = f"{float(parts[8])*100:.1f}%"
                except: pass
                break
except: pass

rows = [
    ['Parameter',          'Value'],
    ['Sample',             'HG00096'],
    ['SRA Accession',      SRA_ID],
    ['Reference',          'hg38 chr22'],
    ['R1 Total Reads',     f"{r1['n']:,}"],
    ['R1 Mean Quality',    f"{r1['mean_q']:.1f}"],
    ['R1 Q30 %',           f"{r1['q30']:.1f}%"],
    ['Total Reads Aligned',total_r],
    ['Mapping Rate',       f"{map_pct}%"],
    ['Duplicate Rate',     dup_pct],
    ['Raw Variants',       f"{len(all_df):,}"],
    ['Filtered Variants',  f"{len(filt_df):,}"],
    ['SNPs',               f"{len(filt_df[filt_df['TYPE']=='SNP']):,}"],
    ['INDELs',             f"{len(filt_df[filt_df['TYPE']=='INDEL']):,}"],
    ['Median Depth',       f"{filt_df['DP'].median():.0f}x"],
]

tbl = ax5.table(cellText=rows[1:], colLabels=rows[0],
                loc='center', cellLoc='left')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.25,1.42)
for j in range(2):
    tbl[0,j].set_facecolor('#1565C0')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(rows)):
    for j in range(2):
        tbl[i,j].set_facecolor('#E8F5E9' if i%2==0 else 'white')
ax5.set_title('Pipeline Summary', fontweight='bold', pad=10)

plt.savefig(f'{PROJECT_DIR}/results/plots/02_Variant_Dashboard.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved → results/plots/02_Variant_Dashboard.png')

---
## 📋 STEP 12 — Final Pipeline Summary Report

In [ ]:
sep = '=' * 62
print(sep)
print('       REAL-WORLD NGS PIPELINE — FINAL REPORT')
print(sep)
print(f'  Date       : {datetime.now().strftime("%Y-%m-%d  %H:%M:%S")}')
print(f'  Sample     : HG00096 (1000 Genomes Project)')
print(f'  SRA ID     : {SRA_ID}')
print(f'  Reference  : hg38 {CHROM}')
print(f'  Pipeline   : BWA-MEM → GATK HaplotypeCaller (Best Practices)')
print(sep)

print('\n📌 QUALITY CONTROL')
for m in [r1, r2]:
    st = '✅ PASS' if m['mean_q']>=25 and m['q30']>=60 else '⚠️  REVIEW'
    print(f"   {m['label']}: meanQ={m['mean_q']:.1f}  Q30={m['q30']:.1f}%  "
          f"GC={m['mean_gc']:.1f}%  [{st}]")

print('\n📌 ALIGNMENT')
try:
    with open(f'{PROJECT_DIR}/results/aligned/{SAMPLE_NAME}_flagstat.txt') as fh:
        for ln in fh:
            print(f'   ├── {ln.strip()}')
except: print('   └── See flagstat file')

print('\n📌 DUPLICATES')
print(f'   └── Duplicate rate : {dup_pct}')

print('\n📌 VARIANT SUMMARY')
print(f'   ├── Raw variants        : {len(all_df):,}')
print(f'   ├── After hard filters  : {len(filt_df):,}')
print(f'   ├── SNPs                : {len(filt_df[filt_df["TYPE"]=="SNP"]):,}')
print(f'   ├── INDELs              : {len(filt_df[filt_df["TYPE"]=="INDEL"]):,}')
print(f'   ├── Median QUAL         : {filt_df["QUAL"].median():.1f}')
print(f'   └── Median depth        : {filt_df["DP"].median():.0f}x')

print('\n📌 OUTPUT FILES')
outputs = [
    ('QC Dashboard',       f'results/plots/01_QC_Dashboard.png'),
    ('Variant Dashboard',  f'results/plots/02_Variant_Dashboard.png'),
    ('Filtered Variants',  f'results/annotated/{SAMPLE_NAME}_variants.csv'),
    ('Alignment BAM',      f'results/dedup/{SAMPLE_NAME}_dedup.bam'),
    ('Flagstat',           f'results/aligned/{SAMPLE_NAME}_flagstat.txt'),
    ('Dup Metrics',        f'results/dedup/{SAMPLE_NAME}_dup_metrics.txt'),
]
for name, path in outputs:
    full = f'{PROJECT_DIR}/{path}'
    ok   = os.path.exists(full)
    size = f'{os.path.getsize(full)/1024:.1f} KB' if ok else 'missing'
    print(f"   {'✅' if ok else '❌'}  {name:25s}: {size}")

print(f'\n{sep}')
print('  ✅  PIPELINE COMPLETE!')
print(sep)

---
## 💾 STEP 13 — Download All Results as ZIP

In [ ]:
from google.colab import files
import zipfile

ZIP = '/content/NGS_Results_NCBI.zip'
print('📦 Packaging results...')

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    include = [
        (f'{PROJECT_DIR}/results/plots/01_QC_Dashboard.png',       'plots/01_QC_Dashboard.png'),
        (f'{PROJECT_DIR}/results/plots/02_Variant_Dashboard.png',  'plots/02_Variant_Dashboard.png'),
        (f'{PROJECT_DIR}/results/annotated/{SAMPLE_NAME}_variants.csv', 'variants/variants_filtered.csv'),
        (f'{PROJECT_DIR}/results/aligned/{SAMPLE_NAME}_flagstat.txt',   'stats/flagstat.txt'),
        (f'{PROJECT_DIR}/results/dedup/{SAMPLE_NAME}_dup_metrics.txt',  'stats/dup_metrics.txt'),
    ]
    for src, dst in include:
        if os.path.exists(src):
            zf.write(src, dst)
            print(f'  ✅ Added: {dst}')
        else:
            print(f'  ⚠️  Skipped (not found): {dst}')

sz = os.path.getsize(ZIP)/1024/1024
print(f'\n📦 ZIP size: {sz:.1f} MB')
print('⬇️  Starting download...')
files.download(ZIP)
print('✅ Download initiated!')

---

## 🎯 Interview Talking Points

| Question | Answer |
|----------|--------|
| **What data did you use?** | Real human WES from NCBI SRA (SRR1518253, 1000 Genomes Project HG00096) |
| **Why chr22 only?** | Chr22 is the smallest autosome — ideal for demos. Same code scales to WGS |
| **What filters did you apply?** | GATK Best Practices hard filters: QD<2, FS>60, MQ<40 for SNPs; QD<2, FS>200 for INDELs |
| **What tools?** | SRA Toolkit → FastQC → Trimmomatic → BWA-MEM → Picard → GATK 4.3 → BCFtools |
| **How reproducible is it?** | Fully reproducible — anyone can run it with the SRA accession and this notebook |
| **How would you scale this?** | Snakemake on HPC with SLURM, or Terra/Google Cloud Life Sciences for cloud |

---

**Tools:** `SRA Toolkit` · `FastQC` · `Trimmomatic` · `BWA-MEM` · `SAMtools` · `Picard 2.27` · `GATK 4.3` · `BCFtools` · `Python/pandas/matplotlib`  
**Data:** NCBI SRA — SRR1518253 (1000 Genomes Project, HG00096)  
**Reference:** hg38 chr22 (UCSC)
